# Auditoría de RouterBench 0-shot

Revisar los artefactos iniciales y su trazabilidad. No entrenar ni evaluar modelos. Las cifras de la propuesta son antecedentes por contrastar.


In [ ]:
from pathlib import Path
import json
import sys
import pandas as pd

root = next(path for path in (Path.cwd(), *Path.cwd().parents) if (path / 'configs/experiment.json').is_file())
sys.path.insert(0, str(root / 'src'))
from routerbench.data import sha256_file
from routerbench.features import FEATURE_NAMES

config = json.loads((root / 'configs/experiment.json').read_text(encoding='utf-8'))
print(config['experiment_id'])


La comprobación se limita a integridad, soporte y separación de prompts. No usar estos datos para escoger hiperparámetros a partir de resultados del test.


In [ ]:
audit = json.loads((root / 'reports/audit.json').read_text(encoding='utf-8'))
assert sha256_file(root / config['dataset']['path']) == config['dataset']['sha256']
print({key: audit[key] for key in ['rows', 'columns', 'models_count', 'evaluations', 'duplicate_extra_rows']})
print(audit['thresholds']['0.75'])


La deduplicación conserva la primera fila original sin consultar scores. Los no elegibles permanecen para la simulación futura de la política completa.


In [ ]:
manifest = json.loads((root / 'data/processed/manifest.json').read_text(encoding='utf-8'))
features = pd.read_parquet(root / 'data/processed/features.parquet')
rows = pd.read_parquet(root / 'data/processed/rows.parquet')
assert tuple(features.columns) == FEATURE_NAMES
assert len(features) == len(rows) == manifest['deduplicated_rows']
assert not features.isna().any().any()
assert rows.groupby('prompt_group')['split'].nunique().max() == 1
for name, checksum in manifest['files_sha256'].items():
    assert sha256_file(root / 'data/processed' / name) == checksum
assert manifest['final_test_evaluated'] is False
print(features.shape)
print(manifest['counts'])


La auditoría nueva obtiene 34,977 elegibles tras deduplicar; la propuesta dice 34,976. Once grupos duplicados tienen resultados distintos. La diferencia se conserva y se explica en `reports/auditoria_inicial.md`.

Siguiente paso: comparar modelos dentro del train y congelar decisiones antes de evaluar desempeño en test. Consultar `docs/protocolo.md`.
